# Deal Analyzer v1 — Prototype Notebook

This notebook is the **Phase 1 prototype** for the real estate deal analyzer.

Inputs:
- Property address
- Asking price
- Hold period
- Strategy
- Memo PDF (optional)

Outputs:
1. Extracted facts from PDF
2. Fair value band (low/base/high)
3. Overpay/fair/underpay flag
4. Scenario table
5. Markdown memo

In [ ]:
# Cell 1: Imports
import sys
sys.path.insert(0, './src')

from extract import extract_text_from_pdf, extract_facts
from scenarios import run_analysis, save_analysis
from memo import generate_memo
from utils import slugify
from normalize import normalize_price, normalize_rent, normalize_address

print("Modules loaded.")

## Cell 2: Set Inputs

Fill in your deal details below.

In [ ]:
# Cell 2: Inputs
ADDRESS = "Example Street, London"
PRICE = 500000  # £500,000
HOLD_YEARS = 5
STRATEGY = "btl_yield"  # Options: btl_yield, refurb_flip, development, hold_long
RENT_ANNUAL = 30000  # £2,500 pcm × 12 = £30,000 pa
SQFT = 750

PDF_PATH = None  # Set to a file path like "data/raw/memo.pdf" if you have one

print(f"Address: {ADDRESS}")
print(f"Price: £{PRICE:,}")
print(f"Hold: {HOLD_YEARS} years")
print(f"Strategy: {STRATEGY}")
print(f"Rent: £{RENT_ANNUAL:,}/year")
print(f"Sqft: {SQFT}")

## Cell 3: Extract PDF (if provided)

In [ ]:
# Cell 3: PDF Extraction
extracted_facts = {}

if PDF_PATH:
    try:
        raw_text = extract_text_from_pdf(PDF_PATH)
        extracted_facts = extract_facts(raw_text)
        print("Extracted facts from PDF:")
        for k, v in extracted_facts.items():
            if v is not None and v != []:
                print(f"  {k}: {v}")
    except Exception as e:
        print(f"PDF extraction failed: {e}")
else:
    print("No PDF provided. Using manual inputs only.")

## Cell 4: Run Valuation

In [ ]:
# Cell 4: Run Analysis
result = run_analysis(
    address=ADDRESS,
    price=PRICE,
    hold_years=HOLD_YEARS,
    strategy=STRATEGY,
    rent_annual=RENT_ANNUAL,
    sqft=SQFT,
    extracted_facts=extracted_facts,
)

print("Analysis complete.")
print("\nValuation:")
for k, v in result['valuation'].items():
    print(f"  {k}: {v}")

print("\nScenario:")
for k, v in result['scenario'].items():
    print(f"  {k}: {v}")

## Cell 5: Generate Memo

In [ ]:
# Cell 5: Generate Memo
memo = generate_memo(result)
print(memo)

## Cell 6: Save to Deals Folder

In [ ]:
# Cell 6: Save
deal_id = slugify(ADDRESS)[:20]
deal_dir = save_analysis(result, deal_id, base_path=".")

# Also save memo as markdown
memo_path = deal_dir / "output.md"
with open(memo_path, "w") as f:
    f.write(memo)

print(f"Saved to: {deal_dir}")
print(f"  input.json, extracted.json, output.json, output.md")